In [13]:
# Core libraries
import json
import pandas as pd
import numpy as np
from datetime import datetime

# Display settings for better inspection
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 200)

In [15]:
def load_jsonl(filepath):
    records = []
    malformed_count = 0
    
    with open(filepath, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError:
                malformed_count += 1
                continue
    
    print(f"✅ Loaded {len(records)} records")
    print(f"⚠️ Skipped {malformed_count} malformed rows")
    
    return records

# Example usage
file_path = "data.jsonl"  # update path
raw_data = load_jsonl(file_path)

✅ Loaded 8799 records
⚠️ Skipped 0 malformed rows


In [17]:
def normalize_data(records):
    df = pd.json_normalize(records, sep='_')
    return df

df_raw = normalize_data(raw_data)

print("Normalized shape:", df_raw.shape)

Normalized shape: (8799, 5235)


In [18]:
def extract_columns(df):
    def safe_col(col, default=np.nan):
        if col in df.columns:
            return df[col]
        else:
            # Return a Series aligned with df index
            return pd.Series([default] * len(df), index=df.index)

    clean_df = pd.DataFrame({
        "subreddit": safe_col("subreddit"),
        "author": safe_col("author"),
        "title": safe_col("title", ""),
        "selftext": safe_col("selftext", ""),
        "created_utc": safe_col("created_utc"),
        "score": safe_col("score", 0),
        "num_comments": safe_col("num_comments", 0),
        "url": safe_col("url"),
        "crosspost_parent": safe_col("crosspost_parent"),
        "num_crossposts": safe_col("num_crossposts", 0),
    })

    return clean_df

In [21]:
def parse_timestamps(df):
    def safe_parse(ts):
        try:
            return datetime.utcfromtimestamp(float(ts))
        except:
            return pd.NaT
    
    df["created_utc"] = df["created_utc"].apply(safe_parse)
    return df

df = parse_timestamps(df)

NameError: name 'df' is not defined